# Churn Prediction — Exploratory Data Analysis

## Objective

The objective of this project is to predict, based on customer features, which customers are likely to churn from a telecom company. My initial assumption is that I would expect a relatively low churn rate, since this is a snapshot of current customers, and most people who were going to churn have likely already left. Given my assumption, I would expect minimizing false negatives to matter most: predicting a customer won't churn when they actually will is more costly to the business (a lost customer with no intervention) than predicting churn for someone who was going to stay (a wasted retention offer). 

In this notebook the objective is to understand the structure, distributions, and relationships in the Teleco Customer Churn dataset to inform cleaning and feature engineering decisions.

## Inputs
- `../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`
- Teleco Customer Churn dataset — 7043 Customers, 21 columns
- Column Description: 
    - **Customer info**: `customerID`, `gender`, `SeniorCitizen`, `Partner`, `Dependents`
    - **Account & billing**: `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`
    - **Services subscribed**: `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
    - **Target**: `Churn`

## Output

## 1.1 Setup and Imports

Importing core libraries for data manipulation, visualization, and analysis. Path constant is defined here to ensure consistent file references throughout the notebook. Warning suppression keeps the output clean.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress outputs of warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

RAW_DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

## 1.2 load dataset

Using pandas to read the dataset and varifying the csv has been properly read into the dataframe with `.shape` and `.head()`. 

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
pd.set_option('display.max_columns', None)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Comparing against what's shown on Kaggle, the shape `(7043, 21)` confirms I've successfully imported 7043 samples and 21 columns, including the target `Churn`. The `.head()` output shows the first 5 rows, and the column names and values match what's described on Kaggle, confirming the data loaded correctly.

## 1.3 Sanity Check

To ensure the data within the dataset is as it should be, before any analysis is done, I will conduct sanity checks. I will check for any incorrect data types, missing values, any impossible or suspicious values, and unexpected categories in order to ensure that analysis can be done without inaccurate or misleading data, and so that down stream models can be trained and tested on good data. 

In [ ]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Looking at the data types of the columns and comparing to what was generated in `.head()`, all columns appear to contain the correct dtype except `TotalCharges`, which is dtype `str`. From `.head()` it looks like it should be `float`, and given that `MonthlyCharges` is `float64` but `TotalCharges` is `str`, this raises the possibility of data entry errors causing the dtype to be `str` instead of `float`. I wil first check for explicit missing values before investigating further.

To check for any missing values and to investigate whether the dtype error in `TotalCharges` is caused by missing or malformed values, I will run `.isnull().sum()` to count `NaN` entries in each column.

In [ ]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

All columns return a sum of 0, meaning there are no explicit `NaN` values in the dataset. However, this does not rule out hidden missing values — since something in `TotalCharges` is causing it to be read as `str` instead of `float`, there may be non-`NaN` placeholder values (such as empty strings or spaces) that `.isnull()` would not catch. I will investigate `TotalCharges` directly to determine what those values are.

To investigate the `TotalCharges` issue, I will convert the column to numeric using `pd.to_numeric()` with `errors='coerce'`, which turns any non-numeric values into `NaN`. I will then create a boolean mask of rows where conversion failed and examine all columns at those rows to identify why these entries are not `float`.

In [ ]:
converted = pd.to_numeric(df['TotalCharges'], errors = 'coerce') 
mask = converted.isna()
print(mask.sum())
df[mask]

11


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


Looking at the features across the 11 flagged rows, the only columns with consistent patterns are `SeniorCitizen`, `Dependents`, `tenure`, and `Churn`. However, `SeniorCitizen` and `Dependents` can be ruled out as meaningful findings — with only 11 rows and these being binary columns, the pattern is likely noise rather than a signal. `tenure`, on the other hand, makes real business sense: it represents how long an individual has been a customer, so a brand new customer (`tenure = 0`) would logically have no `TotalCharges` yet. This also explains why `Churn = No` for all 11 rows — a customer who just signed up cannot have already churned, so this is logically guaranteed rather than an independent finding. The meaningful observation here is `tenure = 0`, which explains the empty `TotalCharges` values. These are new customers with no completed billing cycle, and the blank appears to be a data entry error where `' '` was entered instead of `0.0`.

To confirm exactly what is stored in the `TotalCharges` field for these rows, I will use `repr()` to reveal the raw character content, including any invisible characters like spaces.

In [6]:
repr(df.loc[mask, 'TotalCharges'].iloc[0])

"' '"

The `repr()` output confirms that the 11 flagged `TotalCharges` entries contain a space character `' '` rather than `NaN` or `0.0`. This explains why `.isnull().sum()` returned 0 for `TotalCharges` — pandas does not treat `' '` as a missing value, only explicit `NaN`. These entries will need to be handled in feature engineering by replaing the space and converting the column to `float64`.

Since `TotalCharges` needs to be numeric for accurate analysis, I will convert it in place using `pd.to_numeric()` with `errors='coerce'`, to replace the 11 `' '` entries with `NaN` as a temporary placeholder. I will correctly handle these 11 rows (whether `0.0`, imputation, or removal) in feature engineering — `NaN` is used here to avoid assuming a value before that decision is made.

In [10]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce') 
print(df['TotalCharges'].isnull().sum())
df['TotalCharges'].dtype

11


dtype('float64')

To continue the sanity check, I will run `.describe()` to examine summary statistics for the numeric columns — checking for any impossible values (negatives, zeros where they shouldn't exist) and extreme values that fall far outside the expected range.

In [11]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


`SeniorCitizen` is heavily imbalanced — over 75% of values are 0 (non-senior), meaning the model will see very few examples of senior customers, which could introduce bias toward the majority group. `tenure` has no impossible values — a max of 72 months (~6 years) is a valid tenure for a telecom customer, though the gap between the 75th percentile (55) and max (72) suggests a right-skewed distribution with a long tail, which I expect to see confirmed in univariate analysis. `MonthlyCharges` shows a similar pattern — the min of $18.25 rules out any impossible zero or negative charges, and the gap between the 75th percentile ($89.85) and max ($118.75) again suggests a right tail rather than true outliers. `TotalCharges` follows the same pattern — min of $18.80 rules out impossible values, and the large gap between the 75th percentile ($3,794.73) and max ($8,684.80) suggests a heavily right-skewed distribution, which makes sense since long-tenure customers accumulate significantly more charges. No impossible values found in any numeric column.

To check for unexpected categories or hidden data entry errors across categorical columns, I will run `.describe(include='object')` to get an overview of unique value counts and most frequent values for all `object` columns.

In [12]:
df.describe(include='object')

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,5174


To confirm the exact category labels in columns with 3 unique values, I will run `.value_counts()` on a sample of these columns rather than relying solely on Kaggle's documentation, which labels the third category as `other` without specifying what it represents.

In [6]:
df['InternetService'].value_counts()
df['MultipleLines'].value_counts()
df['OnlineSecurity'].value_counts()


OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

Columns like `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, and `StreamingMovies` each have a third category representing "service not applicable" (e.g., `No internet service`, `No phone service`) 
rather than a simple yes/no choice. This is distinct from `No` — since a customer with no internet service cannot logically have online security, so these are meaningfully different categories. I will decided whether to keep them separate or combine them with `No` in feature engineering after bivariate analysis reveals whether they behave differently with respect to churn.

Summary of sanity checks' issues and observations:

- **`TotalCharges` dtype:** The column was stored as `str` due to 11 rows containing `' '` (a space character) instead of a numeric value. These were temporarily converted to `NaN` using `pd.to_numeric()` so analysis can proceed accurately — the correct handling (replacement with `0.0`, imputation, or removal) will be decided in feature engineering based on the `tenure = 0` pattern found in these rows.
- **Hidden missing values:** `.isnull().sum()` returned 0 for all columns, but the `TotalCharges` investigation revealed that pandas does not treat `' '` as `NaN` — a reminder that zero null counts do not guarantee clean data.
- **`SeniorCitizen` imbalance:** Over 75% of values are 0 (non-senior), which could introduce bias toward the majority group in downstream modeling.
- **No impossible values:** All numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`) have plausible ranges with no negative or zero values where they 
shouldn't exist.
- **Service column third categories:** Columns like `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, and `StreamingMovies` contain a third category representing "service not applicable" rather than a true yes/no choice. Whether to combine this with `No` will be decided in feature engineering after bivariate analysis.